# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook provides a practical workflow for loading, exploring, and processing a Croissant-based dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. You will:

- Load the dataset and explore its Croissant schema/structure
- Inspect available record sets, fields, and columns using their `@id`
- Extract and analyze data using Pandas, referencing only `@id`
- Perform EDA (e.g., filtering, normalization, grouping)
- Visualize the processed data

### Dataset Source
FAIR^2 Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and initialize the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's review the available record sets (and their `@id`), along with the fields (columns) defined for each. All entities will be referenced using their `@id`.

_Note_: `mlcroissant` accesses schema objects via `.metadata.record_sets`, and each record set and field/column has `@id`.

In [ ]:
# List available record sets and their fields, using their @id

record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in the schema.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs['@id']}")
        if 'field' in rs:
            fields = rs['field']
            # field can be a list or a single dict
            if isinstance(fields, dict):
                fields = [fields]
            for field in fields:
                if isinstance(field, dict):
                    print(f"  Field: {field.get('@id', '[no id]')} (col: {field.get('column', '[no column ref]')})")
                else:
                    print(f"  Field ref: {field}")
        else:
            print("  [No fields defined]")

## 3. Data Extraction

Now, load data from the available record set(s) and display the available column `@id`s and a sample of the data. Data extraction uses only the `@id` for record sets and fields.

In [ ]:
# Retrieve all record set @id's as a list from the schema
record_set_ids = []
if not dataset.metadata.record_sets:
    print("No record sets present; cannot extract data.")
else:
    for rs in dataset.metadata.record_sets:
        record_set_ids.append(rs['@id'])

# Extract each record set into a DataFrame using its @id
dataframes = {}
for rs_id in record_set_ids:
    # Use records(record_set=<@id>)
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded records for RecordSet '@id': {rs_id}")
        else:
            print(f"No records found in RecordSet '@id': {rs_id}")
    except Exception as e:
        print(f"Error loading RecordSet {rs_id}: {e}")

# List the column @id's for the first available DataFrame
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nAvailable columns in record set {first_rs_id}:")
    print(list(dataframes[first_rs_id].columns))
    display(dataframes[first_rs_id].head())
else:
    print("No record sets with data could be loaded.")

## 4. Exploratory Data Analysis (EDA)

Choose a numeric field in one of the loaded record sets, filter records, normalize the field, and optionally group by a categorical field. All variables are referenced by their `@id` as per Croissant conventions. Adjust field names and logic if your dataset differs.

In [ ]:
# --- Set up the analysis: choose record set and fields by @id --- #
if dataframes:
    record_set_id = first_rs_id  # Use the first loaded record set
    df = dataframes[record_set_id]
    
    # Identify numeric fields by inspecting column names and datatypes
    print("Available columns and data types in record set:")
    print(df.dtypes)
    
    # Try to find the first numeric column to demonstrate
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric fields found; EDA step will be skipped.")
    else:
        print(f"\nUsing numeric field '@id': {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        
        # Normalize the field
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # If there is a likely grouping field (categorical), try grouping
        # Heuristic: use the first non-numeric field
        group_field_id = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            print(f"\nGrouping by '@id': {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and, if available, show the group-wise means. Plots use the column `@id` as labels.

In [ ]:
if dataframes and numeric_field_id:
    plt.figure(figsize=(7,4))
    plt.hist(df[numeric_field_id].dropna(), bins=20, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    if group_field_id:
        plt.figure(figsize=(8,4))
        plt.bar(grouped_df[group_field_id].astype(str), grouped_df[numeric_field_id], color='orange')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

Using the Croissant schema and `mlcroissant`, we successfully loaded metadata, explored record sets and their structure via `@id`, extracted data for analysis, performed basic EDA steps, and generated a simple visualization. For more information, visit the [mlcroissant documentation](https://mlcroissant.readthedocs.io/), and use the `@id` approach for interoperable, robust data curation and processing.